#### Save the best model

XGBoost was the best model for UCI at all horizons, and best or not significantly different from the best model for PJME
at 24h and 168h (notebook 06). It is also fast and small, so it is used in the prediction page of the dashboard.
Same tuned settings and training data as notebook 04 (PJME: last 4 years of train), seed 0. Saved to models/.

In [1]:
import os
import json

import numpy as np
import pandas as pd
import xgboost as xgb

In [2]:
# why: the dashboard needs a trained model that it can load quickly.
# XGBoost is saved because it was best or equally best in most tests
HORIZONS = [1, 24, 168]
PJME_TRAIN_HOURS = 24 * 365 * 4
os.makedirs("../models", exist_ok=True)

# json.load() reads the best settings saved in notebook 04
with open("../results/baseline_params.json") as f:
    params = json.load(f)

Same features as notebook 04.

In [3]:
# make_features: same function as in notebook 04, builds the lag, rolling mean and calendar columns
# for horizon h. it must be the same, or the saved model would get different inputs
def make_features(df, h):
    X = pd.DataFrame(index=df.index)

    # only lags that are known h hours before, e.g. h = 24 -> 24, 25, ..., 29, 48, 168, 336
    lags = [h + k for k in range(6)] + [m for m in [24, 48, 168, 336] if m >= h]
    for s in sorted(set(lags)):
        # shift(s) = the load from s hours before
        X[f"lag_{s}"] = df["load"].shift(s)

    # average of the last 24 (and 168) known hours
    X["roll_mean_24"] = df["load"].shift(h).rolling(24).mean()
    X["roll_mean_168"] = df["load"].shift(h).rolling(168).mean()

    for c in ["hour", "dayofweek", "month", "dayofyear", "weekend", "holiday"]:
        X[c] = df[c]

    # remove the first rows that have no past values
    X = X.dropna()
    y = df["load"].loc[X.index]
    return X, y

In [4]:
# train one XGBoost model for each dataset and each horizon (6 models) and save them
for name in ["pjme", "uci"]:
    train = pd.read_parquet(f"../data/processed/{name}_train.parquet")
    val = pd.read_parquet(f"../data/processed/{name}_val.parquet")
    test = pd.read_parquet(f"../data/processed/{name}_test.parquet")
    full = pd.concat([train, val, test])

    for h in HORIZONS:
        X, y = make_features(full, h)
        # tr = rows of the train part, te = rows of the test part (True/False masks)
        tr = X.index <= train.index.max()
        if name == "pjme":
            tr = tr & (X.index >= train.index[-PJME_TRAIN_HOURS])
        te = X.index >= test.index.min()

        # same settings and seed as notebook 04, so the model is the same
        model = xgb.XGBRegressor(**params[name]["xgb"], tree_method="hist", n_jobs=-1, random_state=0)
        model.fit(X[tr], y[tr])
        # save_model() writes the trained model to a json file. the dashboard loads it with load_model()
        model.save_model(f"../models/xgb_{name}_h{h}.json")

        # check against the predictions saved in notebook 04
        old = pd.read_parquet(f"../results/predictions/{name}_xgb_h{h}.parquet")["seed_0"]
        # the difference should be 0 (or very close), this shows the saved model is the same as in notebook 04
        new = model.predict(X[te])
        print(f"{name} h={h}: max difference to notebook 04 = {np.abs(new - old.values).max():.6f}")

pjme h=1: max difference to notebook 04 = 0.000000


pjme h=24: max difference to notebook 04 = 0.000000


pjme h=168: max difference to notebook 04 = 0.000000


uci h=1: max difference to notebook 04 = 0.000000


uci h=24: max difference to notebook 04 = 0.000000


uci h=168: max difference to notebook 04 = 0.000000


In [5]:
# list the saved model files
sorted(os.listdir("../models"))

['xgb_pjme_h1.json',
 'xgb_pjme_h168.json',
 'xgb_pjme_h24.json',
 'xgb_uci_h1.json',
 'xgb_uci_h168.json',
 'xgb_uci_h24.json']